# Fairness Auditing

This notebook is implementation-ready for full federated fairness experiments with:

- Two run flags: DRY_RUN and FULL_RUN
- Full 189-run matrix support (baseline + attack-only + attack+defense)
- All three data splits: iid, dirichlet_0_5, strict_non_iid
- All three optimizers: sgd, adamw, fedprox
- All attacks A1-A5 and defenses D1-D5 over valid pairs
- Highly verbose real-time logging (run/round/client/epoch/batch)
- Round-level checkpointing + resume
- One-time model initialization cache (no repeated pretrained loading each round)
- JSON and CSV report artifact export

## How to use

1. Run all setup cells.
2. In the configuration section, keep FULL_RUN=True and FULL_RUN_PROFILE="full_189".
3. Run the execution cell.
4. Review artifacts under artifacts/results and artifacts/checkpoints.

## 1) Set Up Notebook Environment

Install or verify required packages, import core modules, and set reproducibility defaults.

In [ ]:
import importlib
import subprocess
import sys

required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "tqdm": "tqdm",
    "torch": "torch",
    "torchvision": "torchvision",
}

missing = []
for module_name, package_name in required.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        missing.append(package_name)

if missing:
    print(f"Installing missing packages: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All required packages are already installed.")

import json
import math
import os
import random
import re
import time
import traceback
import logging
from copy import deepcopy
from dataclasses import dataclass, asdict, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import models, transforms
from torchvision.datasets import FakeData
from PIL import Image
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Python: {sys.version.split()[0]}")
print(f"Torch: {torch.__version__}")
print(f"Torchvision: {models.__version__ if hasattr(models, '__version__') else 'n/a'}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  cuda:{i} -> {torch.cuda.get_device_name(i)}")

## 2) Define Configuration and Runtime Parameters

Set run flags, full-matrix profile settings, and experiment axes for complete report generation.

In [ ]:
# ===== Required run flags (set exactly one True) =====
DRY_RUN = False
FULL_RUN = True
RESUME_FROM_CHECKPOINT = True

# Runtime policy
DISABLE_RUNTIME_LIMIT = True
RUNTIME_BUDGET_HOURS = 2.0
RUNTIME_BUFFER_MINUTES = 5

# Full-run profile
# full_189: 9 baseline + 45 attack-only + 135 attack+defense = 189
# fast_smoke: tiny profile for quick sanity checks
FULL_RUN_PROFILE = "full_189"

# Global experiment axes (for full_189)
SPLIT_NAMES = ["iid", "dirichlet_0_5", "strict_non_iid"]
OPTIMIZER_NAMES = ["sgd", "adamw", "fedprox"]
ATTR_NAMES = ["race_bin"]
SEED_LIST = [42]

# Base training settings (used for FULL_RUN)
NUM_CLIENTS = 20
CLIENTS_PER_ROUND = 8
ROUNDS = 30
LOCAL_EPOCHS = 1
BATCH_SIZE = 64
MAX_BATCHES_PER_EPOCH = 0  # 0 means no cap

# Dry-run compact settings
DRY_RUN_CLIENTS = 4
DRY_RUN_CLIENTS_PER_ROUND = 4
DRY_RUN_ROUNDS = 2
DRY_RUN_LOCAL_EPOCHS = 1
DRY_RUN_BATCH_SIZE = 16
DRY_RUN_MAX_BATCHES_PER_EPOCH = 3

if DRY_RUN == FULL_RUN:
    raise ValueError("Exactly one of DRY_RUN or FULL_RUN must be True.")


@dataclass
class ExperimentConfig:
    dry_run: bool
    full_run: bool
    resume_from_checkpoint: bool = True

    # Profile
    full_run_profile: str = "full_189"

    # Runtime
    disable_runtime_limit: bool = True
    runtime_budget_hours: float = 2.0
    runtime_buffer_minutes: int = 5

    # Paths
    data_root: Path = Path("data")
    artifacts_root: Path = Path("artifacts")
    checkpoints_root: Path = Path("artifacts/checkpoints")
    logs_root: Path = Path("artifacts/logs")
    results_root: Path = Path("artifacts/results")

    # Data
    image_size: int = 128
    val_split: float = 0.15
    num_workers: int = 4
    pin_memory: bool = True
    strict_real_data: bool = False

    # Federated training
    num_clients: int = 20
    clients_per_round: int = 20
    rounds: int = 150
    local_epochs: int = 5
    batch_size: int = 64
    max_batches_per_epoch: int = 0

    # Optimizer hyper-parameters
    lr_sgd: float = 0.01
    lr_adamw: float = 1e-3
    lr_fedprox: float = 0.01
    momentum_sgd: float = 0.9
    weight_decay: float = 1e-4
    fedprox_mu: float = 1e-3

    # Attack / defense hyper-parameters
    attack_strength: float = 0.25
    defense_strength: float = 0.25

    # Logging / checkpointing
    checkpoint_every_round: int = 5
    log_batch_every: int = 20
    use_amp: bool = True

    # Experiment dimensions
    split_names: List[str] = field(default_factory=lambda: ["iid", "dirichlet_0_5", "strict_non_iid"])
    optimizer_names: List[str] = field(default_factory=lambda: ["sgd", "adamw", "fedprox"])
    attr_names: List[str] = field(default_factory=lambda: ["race_bin"])
    seed_list: List[int] = field(default_factory=lambda: [42])

    # Condition catalog (set by build_config)
    condition_names: List[str] = field(default_factory=list)

    # Dry-run specifics
    dry_run_clients: int = 4
    dry_run_clients_per_round: int = 4
    dry_run_rounds: int = 2
    dry_run_local_epochs: int = 1
    dry_run_batch_size: int = 16
    dry_run_max_batches_per_epoch: int = 3

    def runtime_budget_seconds(self) -> Optional[int]:
        if self.disable_runtime_limit:
            return None
        safe_minutes = max(0, self.runtime_buffer_minutes)
        sec = int(self.runtime_budget_hours * 3600) - safe_minutes * 60
        return max(1, sec)

    def expected_run_count(self) -> int:
        return (
            len(self.condition_names)
            * len(self.split_names)
            * len(self.optimizer_names)
            * len(self.attr_names)
            * len(self.seed_list)
        )


BASELINE_CONDITIONS = ["vanilla"]
ATTACK_ONLY_CONDITIONS = ["A1_only", "A2_only", "A3_only", "A4_only", "A5_only"]
ATTACK_DEFENSE_CONDITIONS = [
    "A1_D1", "A1_D2", "A1_D3",
    "A2_D1", "A2_D2", "A2_D3",
    "A3_D1", "A3_D2", "A3_D3",
    "A4_D3", "A4_D4", "A4_D5",
    "A5_D3", "A5_D4", "A5_D5",
]

FULL_189_CONDITIONS = BASELINE_CONDITIONS + ATTACK_ONLY_CONDITIONS + ATTACK_DEFENSE_CONDITIONS

FULL_RUN_PROFILES: Dict[str, List[str]] = {
    "fast_smoke": ["vanilla", "A1_only"],
    "full_189": FULL_189_CONDITIONS,
}

if FULL_RUN_PROFILE not in FULL_RUN_PROFILES:
    raise ValueError(f"Unknown FULL_RUN_PROFILE={FULL_RUN_PROFILE}. Available={list(FULL_RUN_PROFILES)}")


def build_config(
    dry_run: bool,
    full_run: bool,
    resume_from_checkpoint: bool,
    full_run_profile: str,
) -> ExperimentConfig:
    cfg = ExperimentConfig(
        dry_run=dry_run,
        full_run=full_run,
        resume_from_checkpoint=resume_from_checkpoint,
        full_run_profile=full_run_profile,
        disable_runtime_limit=DISABLE_RUNTIME_LIMIT,
        runtime_budget_hours=RUNTIME_BUDGET_HOURS,
        runtime_buffer_minutes=RUNTIME_BUFFER_MINUTES,
        num_clients=NUM_CLIENTS,
        clients_per_round=CLIENTS_PER_ROUND,
        rounds=ROUNDS,
        local_epochs=LOCAL_EPOCHS,
        batch_size=BATCH_SIZE,
        max_batches_per_epoch=MAX_BATCHES_PER_EPOCH,
        split_names=list(SPLIT_NAMES),
        optimizer_names=list(OPTIMIZER_NAMES),
        attr_names=list(ATTR_NAMES),
        seed_list=list(SEED_LIST),
        dry_run_clients=DRY_RUN_CLIENTS,
        dry_run_clients_per_round=DRY_RUN_CLIENTS_PER_ROUND,
        dry_run_rounds=DRY_RUN_ROUNDS,
        dry_run_local_epochs=DRY_RUN_LOCAL_EPOCHS,
        dry_run_batch_size=DRY_RUN_BATCH_SIZE,
        dry_run_max_batches_per_epoch=DRY_RUN_MAX_BATCHES_PER_EPOCH,
    )

    if cfg.dry_run:
        cfg.num_clients = cfg.dry_run_clients
        cfg.clients_per_round = cfg.dry_run_clients_per_round
        cfg.rounds = cfg.dry_run_rounds
        cfg.local_epochs = cfg.dry_run_local_epochs
        cfg.batch_size = cfg.dry_run_batch_size
        cfg.max_batches_per_epoch = cfg.dry_run_max_batches_per_epoch
        cfg.split_names = ["iid"]
        cfg.optimizer_names = ["sgd"]
        cfg.attr_names = ["race_bin"]
        cfg.seed_list = [42]
        cfg.condition_names = ["vanilla", "A1_only"]
    else:
        cfg.condition_names = list(FULL_RUN_PROFILES[cfg.full_run_profile])

    if cfg.clients_per_round > cfg.num_clients:
        raise ValueError("clients_per_round cannot be greater than num_clients")

    cfg.artifacts_root.mkdir(parents=True, exist_ok=True)
    cfg.checkpoints_root.mkdir(parents=True, exist_ok=True)
    cfg.logs_root.mkdir(parents=True, exist_ok=True)
    cfg.results_root.mkdir(parents=True, exist_ok=True)

    return cfg


CFG = build_config(DRY_RUN, FULL_RUN, RESUME_FROM_CHECKPOINT, FULL_RUN_PROFILE)
print("Config ready:")
print(json.dumps({k: (str(v) if isinstance(v, Path) else v) for k, v in asdict(CFG).items()}, indent=2))
print(f"Full-run profile: {FULL_RUN_PROFILE}")
print(f"Runtime budget (seconds): {CFG.runtime_budget_seconds()}")
print(f"Expected training runs: {CFG.expected_run_count()}")

## 3) Implement Core Data Structures

Define explicit schemas for conditions, per-client metrics, per-round metrics, and checkpoint payloads.

In [ ]:
@dataclass
class AttackDefenseCondition:
    name: str
    attack: str
    defense: str
    category: str
    description: str


CONDITION_REGISTRY: Dict[str, AttackDefenseCondition] = {
    "vanilla": AttackDefenseCondition("vanilla", "none", "none", "baseline", "No attack, no defense"),

    "A1_only": AttackDefenseCondition("A1_only", "A1", "none", "attack_only", "Loss reweighting attack"),
    "A2_only": AttackDefenseCondition("A2_only", "A2", "none", "attack_only", "Gradient-direction bias attack"),
    "A3_only": AttackDefenseCondition("A3_only", "A3", "none", "attack_only", "Fairness-constrained local objective attack"),
    "A4_only": AttackDefenseCondition("A4_only", "A4", "none", "attack_only", "Representation correlation attack"),
    "A5_only": AttackDefenseCondition("A5_only", "A5", "none", "attack_only", "Composite adversarial debiasing attack"),

    "A1_D1": AttackDefenseCondition("A1_D1", "A1", "D1", "attack_defense", "A1 + FAA"),
    "A1_D2": AttackDefenseCondition("A1_D2", "A1", "D2", "attack_defense", "A1 + GSF"),
    "A1_D3": AttackDefenseCondition("A1_D3", "A1", "D3", "attack_defense", "A1 + PFB"),

    "A2_D1": AttackDefenseCondition("A2_D1", "A2", "D1", "attack_defense", "A2 + FAA"),
    "A2_D2": AttackDefenseCondition("A2_D2", "A2", "D2", "attack_defense", "A2 + GSF"),
    "A2_D3": AttackDefenseCondition("A2_D3", "A2", "D3", "attack_defense", "A2 + PFB"),

    "A3_D1": AttackDefenseCondition("A3_D1", "A3", "D1", "attack_defense", "A3 + FAA"),
    "A3_D2": AttackDefenseCondition("A3_D2", "A3", "D2", "attack_defense", "A3 + GSF"),
    "A3_D3": AttackDefenseCondition("A3_D3", "A3", "D3", "attack_defense", "A3 + PFB"),

    "A4_D3": AttackDefenseCondition("A4_D3", "A4", "D3", "attack_defense", "A4 + PFB"),
    "A4_D4": AttackDefenseCondition("A4_D4", "A4", "D4", "attack_defense", "A4 + RF"),
    "A4_D5": AttackDefenseCondition("A4_D5", "A4", "D5", "attack_defense", "A4 + AST"),

    "A5_D3": AttackDefenseCondition("A5_D3", "A5", "D3", "attack_defense", "A5 + PFB"),
    "A5_D4": AttackDefenseCondition("A5_D4", "A5", "D4", "attack_defense", "A5 + RF"),
    "A5_D5": AttackDefenseCondition("A5_D5", "A5", "D5", "attack_defense", "A5 + AST"),
}


VALID_ATTACK_DEFENSE_PAIRS: List[Tuple[str, str]] = [
    ("A1", "D1"), ("A1", "D2"), ("A1", "D3"),
    ("A2", "D1"), ("A2", "D2"), ("A2", "D3"),
    ("A3", "D1"), ("A3", "D2"), ("A3", "D3"),
    ("A4", "D3"), ("A4", "D4"), ("A4", "D5"),
    ("A5", "D3"), ("A5", "D4"), ("A5", "D5"),
]


@dataclass
class RunSpec:
    run_index: int
    run_key: str
    category: str
    condition_name: str
    attack: str
    defense: str
    split_name: str
    optimizer_name: str
    protected_attr: str
    seed: int


@dataclass
class ClientMetrics:
    client_id: int
    loss: float
    accuracy: float
    demographic_parity_gap: float
    equalized_odds_gap: float
    samples: int
    duration_sec: float
    grad_norm_mean: float


@dataclass
class RoundMetrics:
    run_index: int
    run_key: str
    category: str
    condition_name: str
    attack: str
    defense: str
    split_name: str
    optimizer_name: str
    protected_attr: str
    seed: int
    round_idx: int
    selected_clients: List[int]
    aggregate_train_loss: float
    aggregate_train_accuracy: float
    val_loss: float
    val_accuracy: float
    val_demographic_parity_gap: float
    val_equalized_odds_gap: float
    round_duration_sec: float
    elapsed_sec: float
    eta_sec: float
    eta_low_sec: float
    eta_high_sec: float


@dataclass
class RunResult:
    run_index: int
    run_key: str
    category: str
    condition_name: str
    attack: str
    defense: str
    split_name: str
    optimizer_name: str
    protected_attr: str
    seed: int
    started_from_round: int
    finished_round: int
    stopped_reason: str
    total_duration_sec: float
    rounds: List[Dict[str, Any]]


@dataclass
class CheckpointState:
    run_key: str
    round_idx: int
    global_model_state: Dict[str, Any]
    history: List[Dict[str, Any]]
    wallclock_elapsed_sec: float


print(f"Condition registry size: {len(CONDITION_REGISTRY)}")
print(f"Expected full_189 conditions: {len(FULL_189_CONDITIONS)}")
print(f"Current config conditions: {len(CFG.condition_names)}")

## 4) Write the First Working Functions

Implement deterministic setup, verbose logger, adaptive ETA estimator, fairness metrics, and checkpoint primitives.

In [ ]:
def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_available_devices() -> List[torch.device]:
    if torch.cuda.is_available():
        return [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())]
    return [torch.device("cpu")]


def format_seconds(sec: Optional[float]) -> str:
    if sec is None or math.isnan(sec):
        return "unknown"
    sec = max(0.0, float(sec))
    h = int(sec // 3600)
    m = int((sec % 3600) // 60)
    s = int(sec % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


class AdaptiveETAEstimator:
    def __init__(self, alpha: float = 0.30):
        self.alpha = alpha
        self.ewma: Optional[float] = None
        self.history: List[float] = []

    def update(self, duration_sec: float) -> None:
        duration_sec = float(max(0.0, duration_sec))
        self.history.append(duration_sec)
        if self.ewma is None:
            self.ewma = duration_sec
        else:
            self.ewma = self.alpha * duration_sec + (1.0 - self.alpha) * self.ewma

    def estimate(self, remaining_steps: int) -> Tuple[float, float, float]:
        if remaining_steps <= 0:
            return 0.0, 0.0, 0.0

        if not self.history:
            return float("nan"), float("nan"), float("nan")

        base = self.ewma if self.ewma is not None else float(np.mean(self.history))
        median = float(np.median(self.history))
        robust_step = 0.7 * base + 0.3 * median
        eta = robust_step * remaining_steps

        if len(self.history) >= 4:
            q25, q75 = np.quantile(np.array(self.history), [0.25, 0.75])
            eta_low = float(max(0.0, q25 * remaining_steps))
            eta_high = float(max(0.0, q75 * remaining_steps))
        else:
            eta_low = 0.8 * eta
            eta_high = 1.2 * eta

        return float(eta), float(eta_low), float(eta_high)


def build_run_logger(log_root: Path, run_name: str) -> logging.Logger:
    log_root.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger(f"flair_run_{run_name}")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()

    formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

    stream_handler = logging.StreamHandler(sys.stdout)
    stream_handler.setFormatter(formatter)
    logger.addHandler(stream_handler)

    file_path = log_root / f"{run_name}.log"
    file_handler = logging.FileHandler(file_path, mode="a", encoding="utf-8")
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

    logger.propagate = False
    logger.info(f"Logger initialized. File: {file_path}")
    return logger


def _safe_rate(mask: np.ndarray, preds: np.ndarray) -> float:
    if mask.sum() == 0:
        return 0.0
    return float(preds[mask].mean())


def demographic_parity_gap(preds: np.ndarray, protected: np.ndarray) -> float:
    group0 = protected == 0
    group1 = protected == 1
    rate0 = _safe_rate(group0, preds)
    rate1 = _safe_rate(group1, preds)
    return abs(rate0 - rate1)


def equalized_odds_gap(preds: np.ndarray, labels: np.ndarray, protected: np.ndarray) -> float:
    worst = 0.0
    for y in [0, 1]:
        y_mask = labels == y
        g0 = np.logical_and(y_mask, protected == 0)
        g1 = np.logical_and(y_mask, protected == 1)
        r0 = _safe_rate(g0, preds)
        r1 = _safe_rate(g1, preds)
        worst = max(worst, abs(r0 - r1))
    return float(worst)


def accuracy_from_logits(logits: torch.Tensor, labels: torch.Tensor) -> float:
    pred = torch.argmax(logits, dim=1)
    return float((pred == labels).float().mean().item())


class CheckpointManager:
    def __init__(self, checkpoints_root: Path, condition_name: str, logger: logging.Logger):
        self.condition_name = condition_name
        self.logger = logger
        self.dir = checkpoints_root / condition_name
        self.dir.mkdir(parents=True, exist_ok=True)
        self.metrics_jsonl = self.dir / "round_metrics.jsonl"

    def checkpoint_path(self, round_idx: int) -> Path:
        return self.dir / f"round_{round_idx:04d}.pt"

    def _round_from_name(self, path: Path) -> int:
        m = re.search(r"round_(\d+)\.pt", path.name)
        return int(m.group(1)) if m else -1

    def latest_checkpoint(self) -> Optional[Path]:
        candidates = sorted(self.dir.glob("round_*.pt"), key=self._round_from_name)
        return candidates[-1] if candidates else None

    def save(
        self,
        round_idx: int,
        condition_name: str,
        global_model_state: Dict[str, torch.Tensor],
        history: List[Dict[str, Any]],
        wallclock_elapsed_sec: float,
    ) -> Path:
        payload = {
            "condition_name": condition_name,
            "round_idx": round_idx,
            "global_model_state": {k: v.detach().cpu() for k, v in global_model_state.items()},
            "history": history,
            "wallclock_elapsed_sec": float(wallclock_elapsed_sec),
            "saved_at": datetime.now(timezone.utc).isoformat(),
        }
        path = self.checkpoint_path(round_idx)
        torch.save(payload, path)
        self.logger.info(f"[checkpoint] saved -> {path}")
        return path

    def append_round_metrics(self, metrics: Dict[str, Any]) -> None:
        with self.metrics_jsonl.open("a", encoding="utf-8") as f:
            f.write(json.dumps(metrics) + "\n")

    def load_latest(self) -> Optional[CheckpointState]:
        latest = self.latest_checkpoint()
        if latest is None:
            return None
        payload = torch.load(latest, map_location="cpu")
        self.logger.info(f"[checkpoint] loaded <- {latest}")
        return CheckpointState(
            condition_name=payload["condition_name"],
            round_idx=int(payload["round_idx"]),
            global_model_state=payload["global_model_state"],
            history=payload.get("history", []),
            wallclock_elapsed_sec=float(payload.get("wallclock_elapsed_sec", 0.0)),
        )


print("Core utility functions ready.")

## 5) Run an End-to-End Execution Cell

Implement dataset/model/training pipeline, then execute a complete dry run or full run based on the run flags.

In [ ]:
class UTKFaceGenderDataset(Dataset):
    """UTKFace parser using filename pattern: age_gender_race_date.jpg."""

    def __init__(self, image_paths: List[Path], transform: transforms.Compose):
        self.transform = transform
        self.samples: List[Tuple[Path, int, int]] = []

        for p in image_paths:
            parts = p.stem.split("_")
            if len(parts) < 3:
                continue
            try:
                gender = int(parts[1])
                race = int(parts[2])
            except ValueError:
                continue

            label = 1 if gender == 1 else 0
            protected = 0 if race in (0, 1) else 1  # race-only binary protected attribute
            self.samples.append((p, label, protected))

        if not self.samples:
            raise RuntimeError("No valid UTKFace samples found after parsing filenames")

        self.labels_np = np.array([s[1] for s in self.samples], dtype=np.int64)
        self.protected_np = np.array([s[2] for s in self.samples], dtype=np.int64)

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        path, label, protected = self.samples[idx]
        img = Image.open(path).convert("RGB")
        img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.long), torch.tensor(protected, dtype=torch.long)


class SyntheticFairDataset(Dataset):
    """Synthetic fallback dataset for dry-run and functional validation."""

    def __init__(self, size: int, image_size: int, seed: int):
        transform = transforms.Compose(
            [
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ]
        )
        self.base = FakeData(
            size=size,
            image_size=(3, image_size, image_size),
            num_classes=2,
            transform=transform,
            random_offset=seed,
        )
        rng = np.random.default_rng(seed)
        self.labels_np = rng.integers(low=0, high=2, size=size, endpoint=False, dtype=np.int64)
        self.protected_np = rng.integers(low=0, high=2, size=size, endpoint=False, dtype=np.int64)

    def __len__(self) -> int:
        return len(self.base)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        x, _ = self.base[idx]
        y = int(self.labels_np[idx])
        s = int(self.protected_np[idx])
        return x, torch.tensor(y, dtype=torch.long), torch.tensor(s, dtype=torch.long)


def build_transforms(image_size: int) -> transforms.Compose:
    return transforms.Compose(
        [
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )


def discover_utkface_images(data_root: Path) -> List[Path]:
    candidates = [
        data_root / "UTKFace",
        data_root / "utkface",
        data_root,
        Path("UTKFace"),
    ]
    image_paths: List[Path] = []
    exts = ["*.jpg", "*.jpeg", "*.png"]

    for c in candidates:
        if c.exists():
            for ext in exts:
                image_paths.extend(list(c.rglob(ext)))
        if image_paths:
            break

    return sorted(image_paths)


def load_full_dataset(cfg: ExperimentConfig, logger: logging.Logger) -> Tuple[Dataset, str]:
    transform = build_transforms(cfg.image_size)
    image_paths = discover_utkface_images(cfg.data_root)

    if image_paths:
        logger.info(f"Found {len(image_paths)} image files. Attempting UTKFace parsing...")
        try:
            dataset = UTKFaceGenderDataset(image_paths, transform=transform)
            logger.info(f"UTKFace loaded with {len(dataset)} valid samples")
            return dataset, "utkface"
        except Exception as ex:
            if cfg.strict_real_data:
                raise
            logger.warning(f"UTKFace parsing failed ({ex}). Falling back to synthetic.")

    if cfg.strict_real_data:
        raise FileNotFoundError("No valid UTKFace dataset available and strict_real_data=True")

    size = 10000 if cfg.full_run else 1200
    dataset = SyntheticFairDataset(size=size, image_size=cfg.image_size, seed=SEED)
    logger.warning(f"Using synthetic fallback dataset with size={size}")
    return dataset, "synthetic_fallback"


def split_train_val_indices(total_n: int, val_split: float, seed: int) -> Tuple[List[int], List[int]]:
    rng = np.random.default_rng(seed)
    all_indices = np.arange(total_n)
    rng.shuffle(all_indices)
    val_n = max(1, int(total_n * val_split))
    val_indices = all_indices[:val_n].astype(int).tolist()
    train_indices = all_indices[val_n:].astype(int).tolist()
    return train_indices, val_indices


def _rebalance_empty_clients(partitions: Dict[int, List[int]], rng: np.random.Generator) -> Dict[int, List[int]]:
    empty = [cid for cid, idxs in partitions.items() if len(idxs) == 0]
    if not empty:
        return partitions

    donors = sorted(partitions, key=lambda c: len(partitions[c]), reverse=True)
    for cid_empty in empty:
        for donor in donors:
            if len(partitions[donor]) > 1:
                j = int(rng.integers(low=0, high=len(partitions[donor])))
                moved = partitions[donor].pop(j)
                partitions[cid_empty].append(moved)
                break
    return partitions


def make_iid_partitions(train_indices: List[int], num_clients: int, seed: int) -> Dict[int, List[int]]:
    rng = np.random.default_rng(seed)
    shuffled = np.array(train_indices, dtype=np.int64)
    rng.shuffle(shuffled)
    splits = np.array_split(shuffled, num_clients)
    return {cid: split.astype(int).tolist() for cid, split in enumerate(splits)}


def make_dirichlet_partitions(
    train_indices: List[int],
    labels_all: np.ndarray,
    num_clients: int,
    seed: int,
    alpha: float = 0.5,
) -> Dict[int, List[int]]:
    rng = np.random.default_rng(seed)
    partitions: Dict[int, List[int]] = {cid: [] for cid in range(num_clients)}

    train_indices_np = np.array(train_indices, dtype=np.int64)
    labels_train = labels_all[train_indices_np]
    classes = sorted(np.unique(labels_train).tolist())

    for cls in classes:
        cls_indices = train_indices_np[labels_train == cls]
        rng.shuffle(cls_indices)

        proportions = rng.dirichlet(np.full(num_clients, alpha, dtype=np.float64))
        counts = np.floor(proportions * len(cls_indices)).astype(int)

        remainder = len(cls_indices) - int(counts.sum())
        if remainder > 0:
            order = np.argsort(proportions)[::-1]
            for i in range(remainder):
                counts[order[i % num_clients]] += 1

        start = 0
        for cid, cnt in enumerate(counts.tolist()):
            if cnt <= 0:
                continue
            chunk = cls_indices[start : start + cnt].astype(int).tolist()
            partitions[cid].extend(chunk)
            start += cnt

    partitions = _rebalance_empty_clients(partitions, rng)
    for cid in partitions:
        rng.shuffle(partitions[cid])
    return partitions


def make_strict_non_iid_partitions(
    train_indices: List[int],
    labels_all: np.ndarray,
    protected_all: np.ndarray,
    num_clients: int,
    seed: int,
) -> Dict[int, List[int]]:
    rng = np.random.default_rng(seed)
    train_indices_np = np.array(train_indices, dtype=np.int64)

    labels = labels_all[train_indices_np]
    protected = protected_all[train_indices_np]
    combo = labels * 2 + protected

    order = np.argsort(combo)
    sorted_indices = train_indices_np[order]
    shards = np.array_split(sorted_indices, num_clients)

    partitions = {cid: shard.astype(int).tolist() for cid, shard in enumerate(shards)}
    partitions = _rebalance_empty_clients(partitions, rng)
    return partitions


def build_client_partitions(
    split_name: str,
    train_indices: List[int],
    labels_all: np.ndarray,
    protected_all: np.ndarray,
    num_clients: int,
    seed: int,
) -> Dict[int, List[int]]:
    if split_name == "iid":
        return make_iid_partitions(train_indices, num_clients=num_clients, seed=seed)
    if split_name == "dirichlet_0_5":
        return make_dirichlet_partitions(
            train_indices,
            labels_all=labels_all,
            num_clients=num_clients,
            seed=seed,
            alpha=0.5,
        )
    if split_name == "strict_non_iid":
        return make_strict_non_iid_partitions(
            train_indices,
            labels_all=labels_all,
            protected_all=protected_all,
            num_clients=num_clients,
            seed=seed,
        )
    raise ValueError(f"Unknown split_name={split_name}")


def summarize_partition_distribution(
    split_name: str,
    seed: int,
    partitions: Dict[int, List[int]],
    labels_all: np.ndarray,
    protected_all: np.ndarray,
) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    for cid, idxs in partitions.items():
        if idxs:
            idx_np = np.array(idxs, dtype=np.int64)
            y = labels_all[idx_np]
            s = protected_all[idx_np]
            label_pos = float(y.mean())
            protected_pos = float(s.mean())
        else:
            label_pos = 0.0
            protected_pos = 0.0
        rows.append(
            {
                "split_name": split_name,
                "seed": int(seed),
                "client_id": int(cid),
                "samples": int(len(idxs)),
                "label_positive_rate": float(label_pos),
                "protected_positive_rate": float(protected_pos),
            }
        )
    return rows


def save_json(path: Path, payload: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)


_MODEL_INIT_STATE_CACHE: Optional[Dict[str, torch.Tensor]] = None


def _make_model_architecture(num_classes: int = 2) -> nn.Module:
    model = models.resnet18(weights=None)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model


def get_initial_model_state_once(logger: logging.Logger) -> Dict[str, torch.Tensor]:
    global _MODEL_INIT_STATE_CACHE
    if _MODEL_INIT_STATE_CACHE is None:
        logger.info("Loading initial model weights once and caching in memory...")
        try:
            model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
            logger.info("Loaded pretrained ResNet18 weights for one-time initialization.")
        except Exception as ex:
            logger.warning(f"Pretrained weights unavailable ({ex}); using random init once.")
            model = models.resnet18(weights=None)

        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, 2)
        _MODEL_INIT_STATE_CACHE = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    return {k: v.clone() for k, v in _MODEL_INIT_STATE_CACHE.items()}


def instantiate_model(device: torch.device, initial_state: Dict[str, torch.Tensor]) -> nn.Module:
    model = _make_model_architecture(num_classes=2).to(device)
    model.load_state_dict(initial_state)
    return model


def build_optimizer(model: nn.Module, optimizer_name: str, cfg: ExperimentConfig) -> torch.optim.Optimizer:
    if optimizer_name == "sgd":
        return torch.optim.SGD(
            model.parameters(),
            lr=cfg.lr_sgd,
            momentum=cfg.momentum_sgd,
            weight_decay=cfg.weight_decay,
        )
    if optimizer_name == "adamw":
        return torch.optim.AdamW(
            model.parameters(),
            lr=cfg.lr_adamw,
            weight_decay=cfg.weight_decay,
        )
    if optimizer_name == "fedprox":
        return torch.optim.SGD(
            model.parameters(),
            lr=cfg.lr_fedprox,
            momentum=cfg.momentum_sgd,
            weight_decay=cfg.weight_decay,
        )
    raise ValueError(f"Unknown optimizer_name={optimizer_name}")


def _safe_group_mean_tensor(values: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    if bool(torch.any(mask)):
        return values[mask].mean()
    return torch.zeros((), device=values.device, dtype=values.dtype)


def _group_terms(per_sample_loss: torch.Tensor, logits: torch.Tensor, protected: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    g0 = protected == 0
    g1 = protected == 1

    loss_g0 = _safe_group_mean_tensor(per_sample_loss, g0)
    loss_g1 = _safe_group_mean_tensor(per_sample_loss, g1)
    signed_gap = loss_g1 - loss_g0
    abs_gap = torch.abs(signed_gap)

    margin = logits[:, 1] - logits[:, 0]
    protected_pm = protected.float() * 2.0 - 1.0
    corr = torch.mean(margin * protected_pm)

    return loss_g0, loss_g1, signed_gap, abs_gap, corr


def apply_attack_to_loss(
    per_sample_loss: torch.Tensor,
    logits: torch.Tensor,
    labels: torch.Tensor,
    protected: torch.Tensor,
    attack_name: str,
    cfg: ExperimentConfig,
) -> torch.Tensor:
    base = per_sample_loss.mean()
    _, _, signed_gap, abs_gap, corr = _group_terms(per_sample_loss, logits, protected)

    if attack_name == "none":
        return base

    if attack_name == "A1":
        # Group inverse-frequency weighted objective.
        groups, counts = torch.unique(protected, return_counts=True)
        inv = {int(g.item()): 1.0 / max(1, int(c.item())) for g, c in zip(groups, counts)}
        weights = torch.tensor([inv[int(g.item())] for g in protected], device=per_sample_loss.device)
        weights = weights / weights.mean().clamp(min=1e-8)
        return (per_sample_loss * weights).mean()

    if attack_name == "A2":
        return base + cfg.attack_strength * signed_gap

    if attack_name == "A3":
        return base + cfg.attack_strength * abs_gap

    if attack_name == "A4":
        return base + cfg.attack_strength * corr

    if attack_name == "A5":
        composite = 0.5 * signed_gap + 0.5 * abs_gap + 0.5 * corr
        return base + cfg.attack_strength * composite

    raise ValueError(f"Unknown attack_name={attack_name}")


def defense_regularizer(
    per_sample_loss: torch.Tensor,
    logits: torch.Tensor,
    labels: torch.Tensor,
    protected: torch.Tensor,
    defense_name: str,
    cfg: ExperimentConfig,
) -> torch.Tensor:
    if defense_name == "none":
        return torch.zeros((), device=per_sample_loss.device, dtype=per_sample_loss.dtype)

    _, _, signed_gap, abs_gap, corr = _group_terms(per_sample_loss, logits, protected)

    if defense_name == "D1":
        return cfg.defense_strength * abs_gap
    if defense_name == "D2":
        return cfg.defense_strength * (corr ** 2)
    if defense_name == "D3":
        return 0.5 * cfg.defense_strength * abs_gap + 0.5 * cfg.defense_strength * (corr ** 2)
    if defense_name == "D4":
        return cfg.defense_strength * (corr ** 2)
    if defense_name == "D5":
        return cfg.defense_strength * (abs_gap + corr ** 2)

    raise ValueError(f"Unknown defense_name={defense_name}")


def fedprox_regularizer(model: nn.Module, global_params: List[torch.Tensor]) -> torch.Tensor:
    prox = torch.zeros((), device=next(model.parameters()).device)
    for p, g in zip(model.parameters(), global_params):
        prox = prox + torch.sum((p - g) ** 2)
    return prox


def _grad_norm(model: nn.Module) -> float:
    total = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total += float(torch.sum(p.grad.detach() ** 2).item())
    return float(math.sqrt(max(0.0, total)))


def apply_defense_to_gradients(model: nn.Module, defense_name: str) -> float:
    grad_before = _grad_norm(model)

    if defense_name == "none":
        return grad_before

    if defense_name == "D1":
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
    elif defense_name == "D2":
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.5)
        for p in model.parameters():
            if p.grad is not None:
                p.grad.data[torch.abs(p.grad.data) < 1e-5] = 0.0
    elif defense_name == "D3":
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    elif defense_name == "D4":
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.8)
    elif defense_name == "D5":
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.6)
        for p in model.parameters():
            if p.grad is not None:
                p.grad.data.add_(0.001 * torch.randn_like(p.grad.data))
    else:
        raise ValueError(f"Unknown defense_name={defense_name}")

    return grad_before


def local_train_one_client(
    model: nn.Module,
    client_id: int,
    device: torch.device,
    global_state: Dict[str, torch.Tensor],
    train_dataset: Dataset,
    sample_indices: List[int],
    cfg: ExperimentConfig,
    spec: RunSpec,
    logger: logging.Logger,
) -> Tuple[Dict[str, torch.Tensor], ClientMetrics]:
    model.load_state_dict(global_state, strict=True)
    model.train()

    subset = Subset(train_dataset, sample_indices)
    loader = DataLoader(
        subset,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(cfg.pin_memory and device.type == "cuda"),
        drop_last=False,
    )

    optimizer = build_optimizer(model, spec.optimizer_name, cfg)
    criterion = nn.CrossEntropyLoss(reduction="none")

    use_amp = bool(cfg.use_amp and device.type == "cuda")
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    global_params = [p.detach().clone() for p in model.parameters()]

    losses: List[float] = []
    accs: List[float] = []
    grad_norms: List[float] = []

    all_preds: List[np.ndarray] = []
    all_labels: List[np.ndarray] = []
    all_protected: List[np.ndarray] = []

    t0 = time.perf_counter()
    max_batches = cfg.max_batches_per_epoch if cfg.max_batches_per_epoch > 0 else None

    for epoch in range(1, cfg.local_epochs + 1):
        logger.info(
            f"[run={spec.run_index}] [client={client_id}] epoch={epoch}/{cfg.local_epochs} "
            f"batches={len(loader)} attack={spec.attack} defense={spec.defense} opt={spec.optimizer_name}"
        )

        for batch_idx, (x, y, s) in enumerate(loader, start=1):
            if max_batches is not None and batch_idx > max_batches:
                logger.info(
                    f"[run={spec.run_index}] [client={client_id}] max_batches_per_epoch reached ({max_batches})"
                )
                break

            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            s = s.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = model(x)
                per_sample_loss = criterion(logits, y)

                attack_loss = apply_attack_to_loss(
                    per_sample_loss=per_sample_loss,
                    logits=logits,
                    labels=y,
                    protected=s,
                    attack_name=spec.attack,
                    cfg=cfg,
                )

                defense_reg = defense_regularizer(
                    per_sample_loss=per_sample_loss,
                    logits=logits,
                    labels=y,
                    protected=s,
                    defense_name=spec.defense,
                    cfg=cfg,
                )

                prox_reg = torch.zeros((), device=x.device)
                if spec.optimizer_name == "fedprox" or spec.defense == "D3":
                    prox_reg = cfg.fedprox_mu * fedprox_regularizer(model, global_params)

                total_loss = attack_loss + defense_reg + prox_reg

            scaler.scale(total_loss).backward()
            grad_before = apply_defense_to_gradients(model, spec.defense)
            scaler.step(optimizer)
            scaler.update()

            with torch.no_grad():
                batch_acc = accuracy_from_logits(logits, y)
                preds = torch.argmax(logits, dim=1)

            losses.append(float(total_loss.item()))
            accs.append(batch_acc)
            grad_norms.append(float(grad_before))
            all_preds.append(preds.detach().cpu().numpy())
            all_labels.append(y.detach().cpu().numpy())
            all_protected.append(s.detach().cpu().numpy())

            if (batch_idx == 1) or (batch_idx % cfg.log_batch_every == 0):
                logger.info(
                    f"[run={spec.run_index}] [client={client_id}] epoch={epoch} batch={batch_idx} "
                    f"loss={total_loss.item():.5f} acc={batch_acc:.4f} grad_norm={grad_before:.4f}"
                )

    preds_np = np.concatenate(all_preds) if all_preds else np.array([], dtype=np.int64)
    labels_np = np.concatenate(all_labels) if all_labels else np.array([], dtype=np.int64)
    protected_np = np.concatenate(all_protected) if all_protected else np.array([], dtype=np.int64)

    dp_gap = demographic_parity_gap(preds_np, protected_np) if preds_np.size else 0.0
    eo_gap = equalized_odds_gap(preds_np, labels_np, protected_np) if preds_np.size else 0.0

    metrics = ClientMetrics(
        client_id=client_id,
        loss=float(np.mean(losses)) if losses else 0.0,
        accuracy=float(np.mean(accs)) if accs else 0.0,
        demographic_parity_gap=float(dp_gap),
        equalized_odds_gap=float(eo_gap),
        samples=len(subset),
        duration_sec=float(time.perf_counter() - t0),
        grad_norm_mean=float(np.mean(grad_norms)) if grad_norms else 0.0,
    )

    logger.info(
        f"[run={spec.run_index}] [client={client_id}] done "
        f"samples={metrics.samples} loss={metrics.loss:.5f} acc={metrics.accuracy:.4f} "
        f"dp_gap={metrics.demographic_parity_gap:.4f} eo_gap={metrics.equalized_odds_gap:.4f} "
        f"grad_norm={metrics.grad_norm_mean:.4f} duration={format_seconds(metrics.duration_sec)}"
    )

    updated_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    return updated_state, metrics


def fedavg(client_states: List[Dict[str, torch.Tensor]], client_weights: List[float]) -> Dict[str, torch.Tensor]:
    if not client_states:
        raise ValueError("fedavg requires at least one client state")

    w = np.array(client_weights, dtype=np.float64)
    w = w / np.clip(w.sum(), 1e-12, None)

    avg_state: Dict[str, torch.Tensor] = {}
    keys = client_states[0].keys()
    for k in keys:
        accum = None
        for i, st in enumerate(client_states):
            term = st[k].float() * float(w[i])
            accum = term if accum is None else accum + term
        avg_state[k] = accum

    return avg_state


@torch.no_grad()
def evaluate_global_model(model: nn.Module, val_loader: DataLoader, device: torch.device, logger: logging.Logger, run_index: int) -> Dict[str, float]:
    model.eval()
    criterion = nn.CrossEntropyLoss()

    losses: List[float] = []
    all_preds: List[np.ndarray] = []
    all_labels: List[np.ndarray] = []
    all_protected: List[np.ndarray] = []

    for batch_idx, (x, y, s) in enumerate(val_loader, start=1):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        s = s.to(device, non_blocking=True)

        logits = model(x)
        loss = criterion(logits, y)
        preds = torch.argmax(logits, dim=1)

        losses.append(float(loss.item()))
        all_preds.append(preds.detach().cpu().numpy())
        all_labels.append(y.detach().cpu().numpy())
        all_protected.append(s.detach().cpu().numpy())

        if batch_idx == 1:
            logger.info(f"[run={run_index}] [eval] first_batch loss={loss.item():.5f} acc={accuracy_from_logits(logits, y):.4f}")

    preds_np = np.concatenate(all_preds) if all_preds else np.array([], dtype=np.int64)
    labels_np = np.concatenate(all_labels) if all_labels else np.array([], dtype=np.int64)
    protected_np = np.concatenate(all_protected) if all_protected else np.array([], dtype=np.int64)

    val_acc = float((preds_np == labels_np).mean()) if preds_np.size else 0.0
    val_dp = demographic_parity_gap(preds_np, protected_np) if preds_np.size else 0.0
    val_eo = equalized_odds_gap(preds_np, labels_np, protected_np) if preds_np.size else 0.0

    return {
        "val_loss": float(np.mean(losses)) if losses else 0.0,
        "val_accuracy": val_acc,
        "val_demographic_parity_gap": float(val_dp),
        "val_equalized_odds_gap": float(val_eo),
    }


def sanitize_run_key(run_key: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", run_key)


class RunCheckpointManager:
    def __init__(self, checkpoints_root: Path, run_key: str, logger: logging.Logger):
        self.run_key = run_key
        self.logger = logger
        self.dir = checkpoints_root / sanitize_run_key(run_key)
        self.dir.mkdir(parents=True, exist_ok=True)
        self.metrics_jsonl = self.dir / "round_metrics.jsonl"

    def checkpoint_path(self, round_idx: int) -> Path:
        return self.dir / f"round_{round_idx:04d}.pt"

    def _round_from_name(self, path: Path) -> int:
        m = re.search(r"round_(\d+)\.pt", path.name)
        return int(m.group(1)) if m else -1

    def latest_checkpoint(self) -> Optional[Path]:
        candidates = sorted(self.dir.glob("round_*.pt"), key=self._round_from_name)
        return candidates[-1] if candidates else None

    def save(
        self,
        round_idx: int,
        run_key: str,
        global_model_state: Dict[str, torch.Tensor],
        history: List[Dict[str, Any]],
        wallclock_elapsed_sec: float,
    ) -> Path:
        payload = {
            "run_key": run_key,
            "round_idx": int(round_idx),
            "global_model_state": {k: v.detach().cpu() for k, v in global_model_state.items()},
            "history": history,
            "wallclock_elapsed_sec": float(wallclock_elapsed_sec),
            "saved_at": datetime.now(timezone.utc).isoformat(),
        }
        path = self.checkpoint_path(round_idx)
        torch.save(payload, path)
        self.logger.info(f"[checkpoint] saved -> {path}")
        return path

    def append_round_metrics(self, metrics: Dict[str, Any]) -> None:
        with self.metrics_jsonl.open("a", encoding="utf-8") as f:
            f.write(json.dumps(metrics) + "\n")

    def load_latest(self) -> Optional[CheckpointState]:
        latest = self.latest_checkpoint()
        if latest is None:
            return None

        payload = torch.load(latest, map_location="cpu")
        self.logger.info(f"[checkpoint] loaded <- {latest}")
        return CheckpointState(
            run_key=payload["run_key"],
            round_idx=int(payload["round_idx"]),
            global_model_state=payload["global_model_state"],
            history=payload.get("history", []),
            wallclock_elapsed_sec=float(payload.get("wallclock_elapsed_sec", 0.0)),
        )


def make_run_specs(cfg: ExperimentConfig) -> List[RunSpec]:
    specs: List[RunSpec] = []
    run_idx = 1

    for protected_attr in cfg.attr_names:
        for seed in cfg.seed_list:
            for condition_name in cfg.condition_names:
                cond = CONDITION_REGISTRY[condition_name]
                for split_name in cfg.split_names:
                    for optimizer_name in cfg.optimizer_names:
                        run_key = (
                            f"r{run_idx:03d}"
                            f"__{condition_name}"
                            f"__split_{split_name}"
                            f"__opt_{optimizer_name}"
                            f"__attr_{protected_attr}"
                            f"__seed_{seed}"
                        )
                        specs.append(
                            RunSpec(
                                run_index=run_idx,
                                run_key=run_key,
                                category=cond.category,
                                condition_name=condition_name,
                                attack=cond.attack,
                                defense=cond.defense,
                                split_name=split_name,
                                optimizer_name=optimizer_name,
                                protected_attr=protected_attr,
                                seed=seed,
                            )
                        )
                        run_idx += 1

    return specs


def run_single_spec(
    cfg: ExperimentConfig,
    logger: logging.Logger,
    run_name: str,
    spec: RunSpec,
    train_dataset: Dataset,
    val_dataset: Dataset,
    client_partitions: Dict[int, List[int]],
    deadline_ts: Optional[float],
) -> RunResult:
    if spec.condition_name not in CONDITION_REGISTRY:
        raise KeyError(f"Unknown condition: {spec.condition_name}")

    set_all_seeds(spec.seed)

    devices = get_available_devices()
    primary_device = devices[0]

    logger.info(
        f"=== RUN {spec.run_index} | {spec.run_key} | category={spec.category} "
        f"attack={spec.attack} defense={spec.defense} split={spec.split_name} "
        f"optimizer={spec.optimizer_name} seed={spec.seed} ==="
    )

    # Load initial pretrained/random state only once across entire notebook execution.
    initial_state = get_initial_model_state_once(logger)

    global_model = instantiate_model(primary_device, initial_state)
    global_state = {k: v.detach().cpu().clone() for k, v in global_model.state_dict().items()}

    # Reusable client model pool in memory (no weight-download/rebuild per round).
    client_models: Dict[str, nn.Module] = {}
    for d in devices:
        key = str(d)
        client_models[key] = instantiate_model(d, initial_state)

    ckpt_mgr = RunCheckpointManager(cfg.checkpoints_root / run_name, spec.run_key, logger)
    history: List[Dict[str, Any]] = []
    start_round = 1

    if cfg.resume_from_checkpoint:
        ckpt = ckpt_mgr.load_latest()
        if ckpt is not None:
            if ckpt.round_idx >= cfg.rounds:
                logger.info(
                    f"[run={spec.run_index}] already complete in checkpoint (round={ckpt.round_idx}), skipping compute"
                )
                return RunResult(
                    run_index=spec.run_index,
                    run_key=spec.run_key,
                    category=spec.category,
                    condition_name=spec.condition_name,
                    attack=spec.attack,
                    defense=spec.defense,
                    split_name=spec.split_name,
                    optimizer_name=spec.optimizer_name,
                    protected_attr=spec.protected_attr,
                    seed=spec.seed,
                    started_from_round=ckpt.round_idx,
                    finished_round=ckpt.round_idx,
                    stopped_reason="already_completed_checkpoint",
                    total_duration_sec=0.0,
                    rounds=ckpt.history,
                )

            global_state = ckpt.global_model_state
            history = ckpt.history
            start_round = ckpt.round_idx + 1
            logger.info(f"[run={spec.run_index}] resuming from round={start_round}")

    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(cfg.pin_memory and primary_device.type == "cuda"),
    )

    eta = AdaptiveETAEstimator(alpha=0.30)
    wallclock_start = time.time()
    stopped_reason = "completed"

    for round_idx in range(start_round, cfg.rounds + 1):
        if deadline_ts is not None and time.time() >= deadline_ts:
            stopped_reason = "runtime_budget_reached"
            logger.warning(f"[run={spec.run_index}] runtime budget reached, stopping")
            break

        round_t0 = time.perf_counter()

        rng = np.random.default_rng(spec.seed + spec.run_index * 1000 + round_idx)
        selected_clients = rng.choice(
            np.arange(cfg.num_clients),
            size=cfg.clients_per_round,
            replace=False,
        ).tolist()

        logger.info(
            f"[run={spec.run_index}] [round={round_idx}/{cfg.rounds}] selected_clients={selected_clients}"
        )

        local_states: List[Dict[str, torch.Tensor]] = []
        local_weights: List[float] = []
        local_metrics: List[ClientMetrics] = []

        for i, cid in enumerate(selected_clients):
            d = devices[i % len(devices)]
            m = client_models[str(d)]
            st, cm = local_train_one_client(
                model=m,
                client_id=cid,
                device=d,
                global_state=global_state,
                train_dataset=train_dataset,
                sample_indices=client_partitions[cid],
                cfg=cfg,
                spec=spec,
                logger=logger,
            )
            local_states.append(st)
            local_weights.append(float(max(1, cm.samples)))
            local_metrics.append(cm)

        global_state = fedavg(local_states, local_weights)
        global_model.load_state_dict(global_state, strict=True)

        eval_metrics = evaluate_global_model(global_model, val_loader, primary_device, logger, run_index=spec.run_index)

        round_dur = float(time.perf_counter() - round_t0)
        eta.update(round_dur)
        eta_sec, eta_low, eta_high = eta.estimate(cfg.rounds - round_idx)
        elapsed_sec = float(time.time() - wallclock_start)

        rm = RoundMetrics(
            run_index=spec.run_index,
            run_key=spec.run_key,
            category=spec.category,
            condition_name=spec.condition_name,
            attack=spec.attack,
            defense=spec.defense,
            split_name=spec.split_name,
            optimizer_name=spec.optimizer_name,
            protected_attr=spec.protected_attr,
            seed=spec.seed,
            round_idx=round_idx,
            selected_clients=selected_clients,
            aggregate_train_loss=float(np.mean([m.loss for m in local_metrics])) if local_metrics else 0.0,
            aggregate_train_accuracy=float(np.mean([m.accuracy for m in local_metrics])) if local_metrics else 0.0,
            val_loss=eval_metrics["val_loss"],
            val_accuracy=eval_metrics["val_accuracy"],
            val_demographic_parity_gap=eval_metrics["val_demographic_parity_gap"],
            val_equalized_odds_gap=eval_metrics["val_equalized_odds_gap"],
            round_duration_sec=round_dur,
            elapsed_sec=elapsed_sec,
            eta_sec=float(eta_sec),
            eta_low_sec=float(eta_low),
            eta_high_sec=float(eta_high),
        )

        rm_dict = asdict(rm)
        history.append(rm_dict)
        ckpt_mgr.append_round_metrics(rm_dict)

        logger.info(
            f"[run={spec.run_index}] [round={round_idx}] "
            f"train_loss={rm.aggregate_train_loss:.5f} train_acc={rm.aggregate_train_accuracy:.4f} "
            f"val_loss={rm.val_loss:.5f} val_acc={rm.val_accuracy:.4f} "
            f"val_dp={rm.val_demographic_parity_gap:.4f} val_eo={rm.val_equalized_odds_gap:.4f} "
            f"round_time={format_seconds(rm.round_duration_sec)} "
            f"ETA={format_seconds(rm.eta_sec)} [{format_seconds(rm.eta_low_sec)}..{format_seconds(rm.eta_high_sec)}]"
        )

        if (round_idx % cfg.checkpoint_every_round) == 0:
            ckpt_mgr.save(
                round_idx=round_idx,
                run_key=spec.run_key,
                global_model_state=global_state,
                history=history,
                wallclock_elapsed_sec=elapsed_sec,
            )

    finished_round = history[-1]["round_idx"] if history else 0

    return RunResult(
        run_index=spec.run_index,
        run_key=spec.run_key,
        category=spec.category,
        condition_name=spec.condition_name,
        attack=spec.attack,
        defense=spec.defense,
        split_name=spec.split_name,
        optimizer_name=spec.optimizer_name,
        protected_attr=spec.protected_attr,
        seed=spec.seed,
        started_from_round=start_round,
        finished_round=finished_round,
        stopped_reason=stopped_reason,
        total_duration_sec=float(time.time() - wallclock_start),
        rounds=history,
    )


def export_report_artifacts(
    cfg: ExperimentConfig,
    run_name: str,
    run_results: List[Dict[str, Any]],
    fig2_rows: List[Dict[str, Any]],
    logger: logging.Logger,
) -> Dict[str, str]:
    out: Dict[str, str] = {}

    # Fig.2 (data distribution only)
    fig2_csv = cfg.results_root / f"{run_name}_fig2_data_distribution.csv"
    fig2_json = cfg.results_root / f"{run_name}_fig2_data_distribution.json"
    pd.DataFrame(fig2_rows).to_csv(fig2_csv, index=False)
    save_json(fig2_json, {"rows": fig2_rows})
    out["fig2_csv"] = str(fig2_csv)
    out["fig2_json"] = str(fig2_json)

    flat_round_rows: List[Dict[str, Any]] = []
    run_final_rows: List[Dict[str, Any]] = []

    for rr in run_results:
        rounds = rr.get("rounds", [])
        for r in rounds:
            flat_round_rows.append(r)

        if rounds:
            final_round = rounds[-1]
            run_final_rows.append(
                {
                    "run_index": rr["run_index"],
                    "run_key": rr["run_key"],
                    "category": rr["category"],
                    "condition_name": rr["condition_name"],
                    "attack": rr["attack"],
                    "defense": rr["defense"],
                    "split_name": rr["split_name"],
                    "optimizer_name": rr["optimizer_name"],
                    "protected_attr": rr["protected_attr"],
                    "seed": rr["seed"],
                    "stopped_reason": rr["stopped_reason"],
                    "finished_round": rr["finished_round"],
                    "total_duration_sec": rr["total_duration_sec"],
                    "final_val_accuracy": final_round["val_accuracy"],
                    "final_val_demographic_parity_gap": final_round["val_demographic_parity_gap"],
                    "final_val_equalized_odds_gap": final_round["val_equalized_odds_gap"],
                }
            )

    rounds_csv = cfg.results_root / f"{run_name}_all_rounds.csv"
    finals_csv = cfg.results_root / f"{run_name}_all_runs_final.csv"
    pd.DataFrame(flat_round_rows).to_csv(rounds_csv, index=False)
    pd.DataFrame(run_final_rows).to_csv(finals_csv, index=False)
    out["all_rounds_csv"] = str(rounds_csv)
    out["all_runs_final_csv"] = str(finals_csv)

    df_final = pd.DataFrame(run_final_rows)
    if not df_final.empty:
        baseline = df_final[df_final["category"] == "baseline"]
        attack_only = df_final[df_final["category"] == "attack_only"]
        attack_def = df_final[df_final["category"] == "attack_defense"]

        p_baseline = cfg.results_root / f"{run_name}_table2_baseline.csv"
        p_attack_only = cfg.results_root / f"{run_name}_fig1_fig3_fig5_attack_only.csv"
        p_attack_def = cfg.results_root / f"{run_name}_table1_fig4_attack_defense.csv"
        p_fig6 = cfg.results_root / f"{run_name}_fig6_scatter_source.csv"

        baseline.to_csv(p_baseline, index=False)
        attack_only.to_csv(p_attack_only, index=False)
        attack_def.to_csv(p_attack_def, index=False)
        df_final[[
            "condition_name",
            "split_name",
            "optimizer_name",
            "final_val_accuracy",
            "final_val_demographic_parity_gap",
            "final_val_equalized_odds_gap",
        ]].to_csv(p_fig6, index=False)

        out["table2_baseline_csv"] = str(p_baseline)
        out["attack_only_csv"] = str(p_attack_only)
        out["attack_defense_csv"] = str(p_attack_def)
        out["fig6_scatter_csv"] = str(p_fig6)

        count_summary = {
            "baseline": int((df_final["category"] == "baseline").sum()),
            "attack_only": int((df_final["category"] == "attack_only").sum()),
            "attack_defense": int((df_final["category"] == "attack_defense").sum()),
            "total": int(len(df_final)),
        }

        counts_json = cfg.results_root / f"{run_name}_run_count_summary.json"
        save_json(counts_json, count_summary)
        out["run_count_summary_json"] = str(counts_json)

        logger.info(f"Run count summary: {count_summary}")

    return out


def run_full_experiment(cfg: ExperimentConfig, logger: logging.Logger) -> Dict[str, Any]:
    run_name = datetime.now().strftime("full_run_%Y%m%d_%H%M%S")
    logger.info(f"Starting FULL RUN: {run_name}")

    full_dataset, dataset_source = load_full_dataset(cfg, logger)
    labels_all = np.array(full_dataset.labels_np, dtype=np.int64)
    protected_all = np.array(full_dataset.protected_np, dtype=np.int64)

    train_indices, val_indices = split_train_val_indices(len(full_dataset), cfg.val_split, seed=SEED)
    train_dataset = full_dataset
    val_dataset = Subset(full_dataset, val_indices)

    logger.info(
        f"Dataset source={dataset_source} total={len(full_dataset)} train={len(train_indices)} val={len(val_indices)}"
    )

    # Build and cache client partitions for each (split, seed) pair once.
    partition_cache: Dict[Tuple[str, int], Dict[int, List[int]]] = {}
    fig2_rows: List[Dict[str, Any]] = []

    for seed in cfg.seed_list:
        for split_name in cfg.split_names:
            partitions = build_client_partitions(
                split_name=split_name,
                train_indices=train_indices,
                labels_all=labels_all,
                protected_all=protected_all,
                num_clients=cfg.num_clients,
                seed=seed,
            )
            partition_cache[(split_name, seed)] = partitions
            fig2_rows.extend(
                summarize_partition_distribution(
                    split_name=split_name,
                    seed=seed,
                    partitions=partitions,
                    labels_all=labels_all,
                    protected_all=protected_all,
                )
            )

    run_specs = make_run_specs(cfg)
    expected = cfg.expected_run_count()
    if len(run_specs) != expected:
        raise RuntimeError(f"Run catalog mismatch: generated={len(run_specs)} expected={expected}")

    logger.info(f"Run catalog ready: {len(run_specs)} runs")

    budget_sec = cfg.runtime_budget_seconds()
    deadline_ts = None if budget_sec is None else time.time() + budget_sec

    run_results: List[Dict[str, Any]] = []

    for spec in run_specs:
        if deadline_ts is not None and time.time() >= deadline_ts:
            logger.warning("Global runtime budget reached before all runs finished")
            break

        partitions = partition_cache[(spec.split_name, spec.seed)]

        result = run_single_spec(
            cfg=cfg,
            logger=logger,
            run_name=run_name,
            spec=spec,
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            client_partitions=partitions,
            deadline_ts=deadline_ts,
        )
        run_results.append(asdict(result))

    report_files = export_report_artifacts(
        cfg=cfg,
        run_name=run_name,
        run_results=run_results,
        fig2_rows=fig2_rows,
        logger=logger,
    )

    summary = {
        "mode": "full_run",
        "run_name": run_name,
        "dataset_source": dataset_source,
        "config": {k: (str(v) if isinstance(v, Path) else v) for k, v in asdict(cfg).items()},
        "total_runs_generated": len(run_specs),
        "total_runs_executed": len(run_results),
        "runs": run_results,
        "report_files": report_files,
        "finished_at": datetime.now(timezone.utc).isoformat(),
    }

    summary_json = cfg.results_root / f"{run_name}_summary.json"
    save_json(summary_json, summary)
    logger.info(f"Saved full run summary -> {summary_json}")

    return summary


def run_dry_run_suite(cfg: ExperimentConfig, logger: logging.Logger) -> Dict[str, Any]:
    run_name = datetime.now().strftime("dry_run_%Y%m%d_%H%M%S")
    logger.info(f"Starting DRY RUN: {run_name}")

    checks: List[Dict[str, Any]] = []

    def run_check(name: str, fn):
        t0 = time.perf_counter()
        try:
            fn()
            checks.append({
                "name": name,
                "status": "pass",
                "duration_sec": float(time.perf_counter() - t0),
                "error": "",
            })
            logger.info(f"[dry-check] PASS {name}")
        except Exception as ex:
            checks.append({
                "name": name,
                "status": "fail",
                "duration_sec": float(time.perf_counter() - t0),
                "error": f"{type(ex).__name__}: {ex}",
            })
            logger.exception(f"[dry-check] FAIL {name}")

    cache: Dict[str, Any] = {}

    run_check("filesystem_paths", lambda: [
        p.mkdir(parents=True, exist_ok=True)
        for p in [cfg.artifacts_root, cfg.checkpoints_root, cfg.logs_root, cfg.results_root]
    ])

    def _dataset_and_partition_check():
        full_dataset, dataset_source = load_full_dataset(cfg, logger)
        labels_all = np.array(full_dataset.labels_np, dtype=np.int64)
        protected_all = np.array(full_dataset.protected_np, dtype=np.int64)

        train_indices, val_indices = split_train_val_indices(len(full_dataset), cfg.val_split, seed=SEED)
        partitions = build_client_partitions(
            split_name="iid",
            train_indices=train_indices,
            labels_all=labels_all,
            protected_all=protected_all,
            num_clients=cfg.num_clients,
            seed=SEED,
        )

        cache["train_dataset"] = full_dataset
        cache["val_dataset"] = Subset(full_dataset, val_indices)
        cache["partitions"] = partitions
        cache["dataset_source"] = dataset_source

    run_check("dataset_partition_loading", _dataset_and_partition_check)

    def _model_cache_check():
        _ = get_initial_model_state_once(logger)

    run_check("model_weight_cache_once", _model_cache_check)

    def _mini_runs_check():
        mini_cfg = deepcopy(cfg)
        mini_cfg.condition_names = ["vanilla", "A1_only"]
        mini_cfg.split_names = ["iid"]
        mini_cfg.optimizer_names = ["sgd", "adamw", "fedprox"]
        mini_cfg.attr_names = ["race_bin"]
        mini_cfg.seed_list = [42]
        mini_cfg.rounds = min(mini_cfg.rounds, 2)
        mini_cfg.local_epochs = min(mini_cfg.local_epochs, 1)
        mini_cfg.max_batches_per_epoch = min(max(1, mini_cfg.max_batches_per_epoch), 2)

        specs = make_run_specs(mini_cfg)
        deadline_ts = time.time() + 900.0

        for spec in specs[:2]:
            _ = run_single_spec(
                cfg=mini_cfg,
                logger=logger,
                run_name=f"{run_name}_mini",
                spec=spec,
                train_dataset=cache["train_dataset"],
                val_dataset=cache["val_dataset"],
                client_partitions=cache["partitions"],
                deadline_ts=deadline_ts,
            )

    run_check("mini_federated_roundtrip", _mini_runs_check)

    failed = [c for c in checks if c["status"] == "fail"]
    status = "success" if not failed else "failed"

    report = {
        "mode": "dry_run",
        "run_name": run_name,
        "status": status,
        "checks": checks,
        "dataset_source": cache.get("dataset_source", "unknown"),
        "config": {k: (str(v) if isinstance(v, Path) else v) for k, v in asdict(cfg).items()},
        "finished_at": datetime.now(timezone.utc).isoformat(),
    }

    report_path = cfg.results_root / f"{run_name}_report.json"
    save_json(report_path, report)
    logger.info(f"Saved dry-run report -> {report_path}")

    if failed:
        raise RuntimeError(f"Dry run failed with {len(failed)} failed checks")

    return report


print("Full matrix pipeline functions are ready (all attacks, defenses, optimizers, splits).")

In [ ]:
# ==== End-to-end execution cell ====
set_all_seeds(SEED)

run_mode = "dry_run" if CFG.dry_run else "full_run"
run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
logger = build_run_logger(CFG.logs_root, f"{run_mode}_{run_stamp}")

logger.info("============================================================")
logger.info(f"Starting mode={run_mode}")
logger.info(f"Profile={CFG.full_run_profile}")
logger.info(f"Runtime budget seconds={CFG.runtime_budget_seconds()}")
logger.info(f"Conditions={len(CFG.condition_names)}")
logger.info(f"Splits={CFG.split_names}")
logger.info(f"Optimizers={CFG.optimizer_names}")
logger.info(f"Expected runs={CFG.expected_run_count()}")
logger.info("============================================================")

if CFG.dry_run:
    dry_report = run_dry_run_suite(CFG, logger)
    print("Dry run completed successfully.")
    print(json.dumps({
        "status": dry_report["status"],
        "run_name": dry_report["run_name"],
    }, indent=2))
else:
    full_summary = run_full_experiment(CFG, logger)

    print("Full run completed.")
    print(json.dumps({
        "run_name": full_summary["run_name"],
        "total_runs_generated": full_summary["total_runs_generated"],
        "total_runs_executed": full_summary["total_runs_executed"],
        "report_files": full_summary["report_files"],
    }, indent=2))

    if CFG.full_run_profile == "full_189":
        if full_summary["total_runs_executed"] != 189:
            raise RuntimeError(
                f"Expected 189 executed runs for full_189 profile, got {full_summary['total_runs_executed']}"
            )

## 6) Add Basic Assertions and Unit-Style Checks

Validate key invariants, edge cases, and expected failure modes.

In [ ]:
# Configuration invariants
assert CFG.dry_run ^ CFG.full_run, "Exactly one run mode must be active"
assert CFG.clients_per_round <= CFG.num_clients, "clients_per_round cannot exceed num_clients"

unknown_conditions = [c for c in CFG.condition_names if c not in CONDITION_REGISTRY]
assert not unknown_conditions, f"Unknown conditions configured: {unknown_conditions}"

# Run-catalog invariants
specs_preview = make_run_specs(CFG)
assert len(specs_preview) == CFG.expected_run_count(), "Run catalog size mismatch"

if CFG.full_run_profile == "full_189" and CFG.full_run:
    per_axis = len(CFG.split_names) * len(CFG.optimizer_names) * len(CFG.attr_names) * len(CFG.seed_list)
    expected_baseline = 1 * per_axis
    expected_attack_only = 5 * per_axis
    expected_attack_defense = 15 * per_axis
    expected_total = expected_baseline + expected_attack_only + expected_attack_defense

    baseline_count = sum(1 for s in specs_preview if s.category == "baseline")
    attack_only_count = sum(1 for s in specs_preview if s.category == "attack_only")
    attack_defense_count = sum(1 for s in specs_preview if s.category == "attack_defense")

    assert baseline_count == expected_baseline, f"Baseline runs mismatch: {baseline_count} != {expected_baseline}"
    assert attack_only_count == expected_attack_only, f"Attack-only runs mismatch: {attack_only_count} != {expected_attack_only}"
    assert attack_defense_count == expected_attack_defense, f"Attack+Defense runs mismatch: {attack_defense_count} != {expected_attack_defense}"
    assert len(specs_preview) == expected_total, f"Total runs mismatch: {len(specs_preview)} != {expected_total}"

# Fairness metrics edge checks
preds = np.array([0, 1, 1, 0, 1, 0], dtype=np.int64)
labels = np.array([0, 1, 1, 0, 0, 1], dtype=np.int64)
protected = np.array([0, 0, 1, 1, 0, 1], dtype=np.int64)

dp = demographic_parity_gap(preds, protected)
eo = equalized_odds_gap(preds, labels, protected)

assert 0.0 <= dp <= 1.0, "DP gap should be in [0,1]"
assert 0.0 <= eo <= 1.0, "EO gap should be in [0,1]"

# Failure-mode checks
try:
    _ = apply_attack_to_loss(torch.ones(4), torch.ones(4, 2), torch.tensor([0, 1, 0, 1]), torch.tensor([0, 1, 0, 1]), "INVALID", CFG)
    raise AssertionError("Expected ValueError for invalid attack")
except ValueError:
    pass

try:
    dummy = nn.Linear(4, 2)
    _ = apply_defense_to_gradients(dummy, "INVALID")
    raise AssertionError("Expected ValueError for invalid defense")
except ValueError:
    pass

print(f"All notebook assertions/checks passed. Catalog size={len(specs_preview)}")

## 7) Refactor into Reusable Components

Provide reusable presets and launcher wrappers so this notebook can be used as a repeatable execution template.

In [ ]:
def configure_full_profile(cfg: ExperimentConfig, profile: str = "full_189") -> ExperimentConfig:
    cfg = deepcopy(cfg)
    if profile not in FULL_RUN_PROFILES:
        raise ValueError(f"Unknown profile={profile}. Available: {list(FULL_RUN_PROFILES)}")

    cfg.full_run_profile = profile
    cfg.condition_names = list(FULL_RUN_PROFILES[profile])
    return cfg


def configure_unbounded_runtime(cfg: ExperimentConfig) -> ExperimentConfig:
    cfg = deepcopy(cfg)
    cfg.disable_runtime_limit = True
    return cfg


def launch_with_profile(
    profile: str = "full_189",
    dry_run: bool = False,
    full_run: bool = True,
    resume: bool = True,
    disable_runtime_limit: bool = True,
) -> Dict[str, Any]:
    cfg = build_config(
        dry_run=dry_run,
        full_run=full_run,
        resume_from_checkpoint=resume,
        full_run_profile=profile,
    )

    cfg = configure_full_profile(cfg, profile=profile)
    if disable_runtime_limit:
        cfg = configure_unbounded_runtime(cfg)

    run_mode = "dry_run" if cfg.dry_run else "full_run"
    run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    logger = build_run_logger(cfg.logs_root, f"{run_mode}_{run_stamp}_profile_{profile}")

    logger.info(f"Launching profile={profile} mode={run_mode} expected_runs={cfg.expected_run_count()}")
    if cfg.dry_run:
        return run_dry_run_suite(cfg, logger)
    return run_full_experiment(cfg, logger)


print("Reusable profile launchers are ready.")
print("Example usage:")
print("  launch_with_profile(profile='full_189', dry_run=False, full_run=True)")
print("  launch_with_profile(profile='fast_smoke', dry_run=False, full_run=True)")
print("  launch_with_profile(profile='fast_smoke', dry_run=True, full_run=False)")